In [3]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))


In [4]:
import json
from collections import Counter

with open(project_root / "eval" / "eval_set.json") as f:
    raw = json.load(f)

eval_set = raw["questions"]

print(f"{len(eval_set)} questions loaded")
print(Counter(item["question_type"] for item in eval_set))

63 questions loaded
Counter({'single_document': 51, 'cross_document': 12})


In [5]:
import random
random.seed(42)  

from collections import defaultdict
by_type = defaultdict(list)
for item in eval_set:
    by_type[item["question_type"]].append(item)

# keep ALL cross-document questions — there are only 12, and they're
# the most valuable, hardest-won questions in the set (genuine multi-hop
# reasoning grounded in real citation relationships you deliberately built)
TARGET_TOTAL = 24
cross_doc = by_type["cross_document"]  # keep all 12
remaining_budget = TARGET_TOTAL - len(cross_doc)

single_doc = by_type["single_document"]
sampled_single = random.sample(single_doc, min(remaining_budget, len(single_doc)))

reduced_eval_set = cross_doc + sampled_single
print(f"Reduced from {len(eval_set)} to {len(reduced_eval_set)} questions")
print(Counter(item["question_type"] for item in reduced_eval_set))

Reduced from 63 to 24 questions
Counter({'cross_document': 12, 'single_document': 12})


In [6]:
with open(project_root / "eval" / "eval_set_reduced.json", "w") as f:
    json.dump({"metadata": raw["metadata"], "questions": reduced_eval_set}, f, indent=2)